## Setup

### General Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score
from sklearn.impute import SimpleImputer

### Dataset

Sensitive attributes include:
- sex
- race

Target variable:
- two_year_recid

In [ ]:
scores_two_years = pd.read_csv("data/compas-scores-two-years.csv")

columns_to_include = [
    "sex",
    "age",
    "age_cat",
    "race",
    "juv_fel_count",
    "juv_misd_count",
    "juv_other_count",
    "priors_count",
    "c_charge_degree",
    "c_charge_desc",
    "r_charge_degree",
    "r_charge_desc",
    "two_year_recid",
]

df = scores_two_years[columns_to_include]

n = df.shape[0]

main_features = ["sex", "race", "two_year_recid"]

In [ ]:
categorical_columns = [
    "sex",
    "c_charge_degree",
    "two_year_recid",
    "age_cat",
    "race",
    "c_charge_desc",
    "r_charge_degree",
    "r_charge_desc",
]

In [ ]:
def display_main_features_info(df: pd.DataFrame):
    about_main_feats = {}

    for column in main_features:
        temp = pd.DataFrame()

        temp["value"] = df[column].unique().tolist()
        temp["count"] = df[column].value_counts().tolist()
        temp["percentage"] = round(
            df[column].value_counts(normalize=True) * 100, 1
        ).tolist()
        about_main_feats[column] = temp
        print(f"\n\nfeature: {column}")
        print(about_main_feats[column])

## Data Preprocessing

In [ ]:
# df.info()

In [ ]:
# TODO: Handle missing values in "r_charge_degree" and "r_charge_desc" (<50% missing)
# TODO: Handle missing values in "c_charge_desc" (>1% missing)

# Encode categorical variables
df_encoded = df.copy()

for col in df_encoded.columns:
    if df_encoded[col].dtype == "object":
        df_encoded[col] = LabelEncoder().fit_transform(df_encoded[col])

# Models

### Settings

In [ ]:
synthetic_data_count = 5000
num_epochs = 200

### Model Imports

In [ ]:
from ctgan import CTGAN
from ctgan import TVAE
from tabfairgan import TFG
from fairgan import FairGAN

## CTGAN
The CTGAN model learns the joint distribution of the dataset.

### Model Training

In [ ]:
import os

os.environ["LOKY_MAX_CPU_COUNT"] = "1"

In [ ]:
ctgan_df = df.copy()

ctgan_model = CTGAN(epochs=num_epochs, verbose=True)
ctgan_model.fit(ctgan_df, discrete_columns=categorical_columns)

### Synthetic Data Generation

In [ ]:
ctgan_synthetic_data = ctgan_model.sample(synthetic_data_count)

ctgan_synthetic_data.head()

Export the synthetic data for analysis

In [ ]:
ctgan_synthetic_data.to_csv("ctgan_synthetic.csv", index=False)

## TVAE

TVAE learns a latent representation of the data and reconstructs it through a decoder network.

Unlike CTGAN, TVAE optimizes a probabilistic objective rather than using adversarial training.

### Model Training

In [ ]:
tvae_df = df.copy()

tvae_model = TVAE(epochs=num_epochs, verbose=True)
tvae_model.fit(tvae_df, discrete_columns=categorical_columns)

### Synthetic Data Generation

In [ ]:
tvae_synthetic_data = tvae_model.sample(synthetic_data_count)

tvae_synthetic_data.head()

Export the synthetic data for analysis

In [ ]:
tvae_synthetic_data.to_csv("tvae_synthetic.csv", index=False)

## FairGAN

Our implementation of the FairGAN model can be found [here](fairgan.py)

### Model Training

In [ ]:
fairgan_df = df.copy()


# TODO: Make sure race can be handled
fairgan_model = FairGAN(
    fairgan_df, "sex", "two_year_recid", epochs=num_epochs, verbose=True
)
fairgan_model.train()

In [ ]:
tvae_df = df.copy()

tvae_model = TVAE(epochs=num_epochs, verbose=True)
tvae_model.fit(tvae_df, discrete_columns=categorical_columns)

### Synthetic Data Generation

In [ ]:
fairgan_synthetic_data = fairgan_model.generate(synthetic_data_count)

fairgan_synthetic_data.head()

Export the synthetic data for analysis

In [ ]:
fairgan_synthetic_data.to_csv("fairgan_synthetic.csv", index=False)

## TabFairGAN

TabFairGAN differs from standard generative models such as CTGAN in that fairness constraints are incorporated directly into the training process.

Instead of using a separate fit function, the model is trained using a custom training loop.

### Model Training

**Fairness Constraints**

To generate fair synthetic data, you need to pass a dictionary containing the following parameters:

- `fair_epochs`: Number of fair training epochs (integer).
- `lamda`: Lambda parameter controlling the trade-off between fairness and accuracy (float).
- `S`: Protected attribute (string, e.g., "sex").
- `Y`: Decision label (string, e.g., "income").
- `S_under`: Value representing the underprivileged group for the protected attribute (string, e.g., " Female").
- `Y_desire`: Desired value for the label (string, e.g., " >50K").

In [ ]:
tfg_df = df.copy()

# Define fairness configuration
fairness_config = {
    "fair_epochs": 50,
    "lamda": 0.5,
    "S": "sex",
    "Y": "income",
    "S_under": "Female",
    "Y_desire": ">50K",
}

# Initialize TabFairGAN with fairness constraints
tfg_model = TFG(
    tfg_df, epochs=200, batch_size=256, fairness_config=fairness_config
)

# Train the model
tfg_model.train()

### Synthetic Data Generation

In [ ]:
tfg_synthetic_data = tfg_model.generate_fake_df(num_rows=synthetic_data_count)

tfg_synthetic_data.head()

Export the synthetic data for analysis

In [ ]:
tfg_synthetic_data.to_csv("tabfairgan_synthetic.csv", index=False)